# *ASSIGNMENT-0*

In [ ]:
import pandas as pd
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
import random
import numpy as np
from collections import Counter

In [ ]:
BASE_DIR = Path().resolve().parent.parent
DATASET_PATH = BASE_DIR / "data" / "anime_info.csv"
df = pd.read_csv(DATASET_PATH)

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

In [ ]:
print("MISSING VALUES")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])

In [ ]:
print("SAMPLE ANIME")
sample = df[
    [
        "title",
        "genres",
        "themes",
        "demographics",
        "score",
        "members",
    ]
].head(10)
display(sample)

In [ ]:
print("TYPE DISTRIBUTION")
display(df["type"].value_counts())

In [ ]:
print("DEMOGRAPHICS")
display(df["demographics"].value_counts(dropna=False))

In [ ]:
print("TOP 20 BY MEMBERS")
top_members = (
    df[["title", "members"]]
    .sort_values("members", ascending=False)
    .head(20)
)
display(top_members)

In [ ]:
print("TOP 20 BY SCORE")
top_score = (
    df[df["score"].notna()][["title", "score"]]
    .sort_values("score", ascending=False)
    .head(20)
)
display(top_score)

In [ ]:
print("MOST COMMON GENRES")
genre_counter = {}
for genres in df["genres"].dropna():
    for genre in str(genres).split("|"):
        genre = genre.strip()
        if genre:
            genre_counter[genre] = genre_counter.get(genre, 0) + 1
genre_df = (
    pd.DataFrame(genre_counter.items(), columns=["genre", "count"])
    .sort_values("count", ascending=False)
)
display(genre_df.head(20))

## FILTER

In [ ]:
filtered = df[
    (df["members"] >= 50000)
    & (df["genres"].notna())
]
filtered = filtered.drop_duplicates(subset="mal_id")
print(f"Filtered rows: {filtered.shape[0]}")
print(f"Non-filtered rows: {df.shape[0]}")

In [ ]:
G = nx.Graph()
for _, row in filtered.iterrows():
    G.add_node(row["mal_id"], title=row["title"])

In [ ]:
print(G.number_of_nodes())
print(G.number_of_edges())

In [ ]:
for (_, a), (_, b) in combinations(filtered.iterrows(), 2):
    genres_a = set(str(a["genres"]).split("|"))
    genres_b = set(str(b["genres"]).split("|"))
    shared = genres_a & genres_b
    if len(shared) >= 2:
        G.add_edge(a["mal_id"], b["mal_id"])

In [ ]:
plt.figure(figsize=(10,10))
nx.draw_networkx(G, with_labels=False, node_size=5)
plt.show()

## ANALYSIS

### Metrics

In [ ]:
density = nx.density(G)
average_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
components = nx.number_connected_components(G)
assortativity = nx.degree_assortativity_coefficient(G)
clustering = nx.average_clustering(G, nodes=list(G.nodes())[:500])

In [ ]:
print(f"Density: {density:.6f}")
print(f"Average degree: {average_degree:.2f}")
print(f"Connected components: {components}")
print(f"Assortativity: {assortativity:.6f}")
print(f"Average clustering coefficient: {clustering:.6f}")

### Largest connected component

In [ ]:
largest = max(nx.connected_components(G), key=len)
H = G.subgraph(largest).copy()
diameter = nx.diameter(H)
sample_size = min(50, H.number_of_nodes())
sample_nodes = random.sample(list(H.nodes()), sample_size)
path_lengths = []
for node in sample_nodes:
    lengths = nx.single_source_shortest_path_length(H, node)
    path_lengths.extend(lengths.values())
average_path = sum(path_lengths) / len(path_lengths)

print(f"Diameter: {diameter}")
print(f"Estimated average shortest path: {average_path:.3f}")

### Degree centrality

In [ ]:
degree = nx.degree_centrality(G)
top_nodes = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:10]
centrality = pd.DataFrame(
    [
        (G.nodes[node]["title"], value)
        for node, value in top_nodes
    ],
    columns=["Anime", "Degree Centrality"]
)
display(centrality)

### Degree distribution

In [ ]:
degrees = [d for _, d in G.degree()]
plt.figure(figsize=(8,5))
plt.hist(degrees, bins=50, edgecolor="black")
plt.title("Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

In [ ]:
degrees = [d for _, d in G.degree()]
degree_counts = Counter(degrees)
x = sorted(degree_counts.keys())
y = [degree_counts[k] for k in x]

plt.figure()
plt.plot(x, y, marker='o', linestyle='None')
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.title("Degree Distribution")
plt.show()

In [ ]:
x_log = np.log10(x)
y_log = np.log10(y)

plt.figure()
plt.plot(x_log, y_log, marker='o', linestyle='None')
plt.xlabel("log10(Degree)")
plt.ylabel("log10(Frequency)")
plt.title("Degree Distribution (Log-Log)")
plt.show()

In [ ]:
print("Max degree:", max(degrees))
print("Min degree:", min(degrees))
print("Mean degree:", np.mean(degrees))
print("Median degree:", np.median(degrees))